# SwahiliNLU — Baseline Experiments (Modular)
> **Paper:** SwahiliNLU: A Broad-Domain Intent and Slot-Filling Dataset with Pragmatic Tone Annotations
> **Structure:** Core logic lives in `.py` modules written to `/kaggle/working/`.
> The notebook calls them as clean, high-level functions.

---
## Modules
- `swahili_data.py` — data loading and cleaning
- `swahili_datasets.py` — PyTorch Dataset classes
- `swahili_train.py` — fine-tuning and slot filling
- `swahili_llm.py` — zero-shot LLM evaluation
- `swahili_eval.py` — per-domain analysis, charts, LaTeX

---
## Table of Contents
1. [Install & Imports](#1)
2. [Write Modules](#2)
3. [Load Data](#3)
4. [Split](#4)
5. [Intent Classification](#5)
6. [Tone Classification](#6)
7. [Slot Filling](#7)
8. [Zero-shot LLM](#8)
9. [Per-Domain Analysis](#9)
10. [Results & LaTeX](#10)

---
## 1. Install & Imports <a id='1'></a>

In [ ]:
!pip install -q transformers datasets seqeval accelerate anthropic openai google-generativeai

In [ ]:
import json, glob, re, warnings, os, gc, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi':120,'figure.facecolor':'white',
    'axes.facecolor':'#F9F9F7','axes.spines.top':False,
    'axes.spines.right':False,'font.family':'DejaVu Sans',
    'axes.grid':True,'grid.alpha':0.3,
})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED   = 42
torch.manual_seed(SEED); np.random.seed(SEED)
MODELS = {
    'mBERT'   :'bert-base-multilingual-cased',
    'XLM-R'   :'xlm-roberta-base',
    'AfroXLMR':'Davlan/afro-xlmr-base',
}
DATA_DIR = '/kaggle/input/datasets/alfredkondoro/intent-dataset/intent'
print(f'Device: {DEVICE}  |  GPU: {torch.cuda.get_device_name(0) if DEVICE=="cuda" else "N/A"}')

---
## 2. Write Modules <a id='2'></a>
> Run once to write all `.py` files to `/kaggle/working/`.

In [ ]:
%%writefile /kaggle/working/swahili_data.py
"""Data loading utilities for SwahiliNLU."""
import json, glob, re
from pathlib import Path
import pandas as pd

VALID_TONES = {'imperative', 'polite', 'conversational'}

def _extract_tone(slots_str):
    try:
        d = json.loads(str(slots_str))
        return d.get('tone', None)
    except:
        return None

def _clean_slots(slots_str):
    for attempt in [str(slots_str), str(slots_str).replace('""','"')]:
        try:
            d = json.loads(attempt)
            d.pop('tone', None)
            return json.dumps(d)
        except:
            pass
    return str(slots_str)

def parse_line(line):
    """Parse one CSV data line — handles plain and double-escaped JSON."""
    line = line.strip().rstrip(',')
    if not line: return None
    js = line.find('{')
    je = line.rfind('}')
    if js == -1:
        parts = line.split(',', 2)
        if len(parts) < 2: return None
        return {'intent':parts[0].strip(),'utterance':parts[1].strip(),
                'slots':'','tone':parts[2].strip() if len(parts)>2 else None}
    before     = line[:js]
    slots_json = line[js:je+1]
    after      = line[je+1:].strip().strip('"').strip(',').strip()
    parts      = [p.strip().strip('"') for p in before.rstrip(',"').split(',',1)]
    if len(parts) < 2: return None
    slots_fixed = slots_json.replace('""','"')
    tone = None
    if after and after.lower() in ('imperative','polite','conversational'):
        tone = after.lower()
    if not tone:
        for attempt in [slots_fixed, slots_json]:
            try:
                d = json.loads(attempt)
                if d.get('tone'): tone = d['tone']; break
            except: pass
    if not tone:
        m = re.search(r'"tone"\s*:\s*"([a-z]+)"', slots_json, re.I)
        if m and m.group(1).lower() in VALID_TONES: tone = m.group(1).lower()
    slots_clean = '{}'
    for attempt in [slots_fixed, slots_json]:
        try:
            d = json.loads(attempt); d.pop('tone',None)
            slots_clean = json.dumps(d); break
        except: pass
    return {'intent':parts[0].strip(),'utterance':parts[1].strip(),
            'slots':slots_clean,'tone':tone}

def load_file(f):
    """Load one domain CSV — returns a DataFrame."""
    fname = Path(f).name
    rows  = []
    with open(f,'r',encoding='utf-8-sig') as fh:
        lines = fh.readlines()
    for line in lines[1:]:
        parsed = parse_line(line)
        if parsed:
            parsed['source_file'] = fname
            rows.append(parsed)
    return pd.DataFrame(rows) if rows else pd.DataFrame()

def load_dataset(data_dir):
    """Load and clean all domain CSVs. Returns a clean DataFrame."""
    import glob as _glob
    csv_files = sorted(_glob.glob(f'{data_dir}/*.csv'))
    frames = [load_file(f) for f in csv_files]
    df = pd.concat([fr for fr in frames if len(fr)>0], ignore_index=True)
    df['tone']      = df['tone'].astype(str).str.strip().str.lower().str.replace('"','',regex=False)
    df['slots']     = df['slots'].astype(str).str.strip()
    df['utterance'] = df['utterance'].astype(str).str.strip().str.strip(',"')
    before = len(df)
    df = df[df['tone'].isin(VALID_TONES)]
    df = df.dropna(subset=['intent','utterance'])
    df = df[df['utterance'].str.strip()!='']
    df = df.reset_index(drop=True)
    print(f'Loaded {len(df):,} rows from {df["source_file"].nunique()} domains '
          f'(dropped {before-len(df)} invalid rows)')
    return df


In [ ]:
%%writefile /kaggle/working/swahili_datasets.py
"""PyTorch Dataset classes for intent, tone and slot tasks."""
import json
import torch
from torch.utils.data import Dataset

class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(list(texts),truncation=True,padding=True,
                                   max_length=max_len,return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k:v[idx] for k,v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

class SlotDataset(Dataset):
    def __init__(self, data_df, tokenizer, tag2id, max_len=128):
        self.tokenizer = tokenizer
        self.tag2id    = tag2id
        self.max_len   = max_len
        self.data      = data_df.reset_index(drop=True)
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        tokens = self.data.loc[idx,'tokens']
        tags   = self.data.loc[idx,'tags']
        enc    = self.tokenizer(tokens,is_split_into_words=True,
                                truncation=True,padding='max_length',
                                max_length=self.max_len,return_tensors='pt')
        word_ids  = enc.word_ids(batch_index=0)
        label_ids = []
        prev_word = None
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)
            elif wid != prev_word:
                label_ids.append(self.tag2id.get(tags[wid],0))
            else:
                tag = tags[wid]
                if tag.startswith('B-'): tag='I-'+tag[2:]
                label_ids.append(self.tag2id.get(tag,0))
            prev_word = wid
        return {'input_ids':enc['input_ids'].squeeze(),
                'attention_mask':enc['attention_mask'].squeeze(),
                'labels':torch.tensor(label_ids,dtype=torch.long)}

def make_bio_tags(utterance, slots_str):
    """Convert utterance + slots JSON to BIO tag list."""
    tokens = utterance.strip().split()
    tags   = ['O'] * len(tokens)
    try:
        slots = json.loads(slots_str) if slots_str not in ('','{}','nan') else {}
    except:
        slots = {}
    for slot_name, slot_value in slots.items():
        if not slot_value: continue
        sv_toks = str(slot_value).strip().split()
        n = len(sv_toks)
        ul = [t.lower() for t in tokens]
        sl = [t.lower() for t in sv_toks]
        for i in range(len(tokens)-n+1):
            if ul[i:i+n]==sl:
                tags[i]=f'B-{slot_name}'
                for j in range(1,n): tags[i+j]=f'I-{slot_name}'
                break
    return tokens, tags

def build_bio_df(source_df):
    """Build a BIO-tagged DataFrame from a source DataFrame."""
    import pandas as pd
    rows = []
    for _,row in source_df.iterrows():
        t,g = make_bio_tags(row['utterance'],row['slots'])
        rows.append({'tokens':t,'tags':g})
    return pd.DataFrame(rows)


In [ ]:
%%writefile /kaggle/working/swahili_train.py
"""Fine-tuning utilities for classification and slot filling tasks."""
import gc
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report, confusion_matrix)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           AutoModelForTokenClassification,
                           TrainingArguments, Trainer,
                           DataCollatorForTokenClassification)
from seqeval.metrics import f1_score as seq_f1
from seqeval.metrics import classification_report as seq_report
from swahili_datasets import IntentDataset, SlotDataset, build_bio_df


def train_classifier(model_name, model_key, train_df, dev_df, test_df,
                     label_encoder, num_labels, task_name,
                     device='cuda', seed=42, max_len=128):
    """Fine-tune a sequence classification model. Returns overall and per-domain results."""
    print(f'\n{"="*60}\n  {task_name} | {model_key}\n{"="*60}')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    col = 'intent' if task_name=='Intent' else 'tone'
    train_labels = label_encoder.transform(train_df[col])
    dev_labels   = label_encoder.transform(dev_df[col])
    test_labels  = label_encoder.transform(test_df[col])
    train_ds = IntentDataset(train_df['utterance'],train_labels,tokenizer,max_len)
    dev_ds   = IntentDataset(dev_df['utterance'],  dev_labels,  tokenizer,max_len)
    test_ds  = IntentDataset(test_df['utterance'], test_labels, tokenizer,max_len)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,num_labels=num_labels,ignore_mismatched_sizes=True).to(device)

    def compute_metrics(eval_pred):
        logits,labels = eval_pred
        preds = np.argmax(logits,axis=-1)
        return {'accuracy':accuracy_score(labels,preds),
                'macro_f1':f1_score(labels,preds,average='macro',zero_division=0)}

    args = TrainingArguments(
        output_dir=f'./results/{task_name.lower()}_{model_key}',
        num_train_epochs=10,per_device_train_batch_size=32,
        per_device_eval_batch_size=64,warmup_ratio=0.1,weight_decay=0.01,
        learning_rate=2e-5,eval_strategy='epoch',save_strategy='no',
        load_best_model_at_end=False,logging_steps=20,seed=seed,
        report_to='none',fp16=(device=='cuda'),
    )
    trainer = Trainer(model=model,args=args,train_dataset=train_ds,
                      eval_dataset=dev_ds,compute_metrics=compute_metrics)
    trainer.train()

    preds_out  = trainer.predict(test_ds)
    preds      = np.argmax(preds_out.predictions,axis=-1)
    pred_names = label_encoder.inverse_transform(preds)
    true_names = label_encoder.inverse_transform(test_labels)
    acc = accuracy_score(test_labels,preds)
    mf1 = f1_score(test_labels,preds,average='macro',zero_division=0)
    wf1 = f1_score(test_labels,preds,average='weighted',zero_division=0)
    print(f'\n  Accuracy {acc:.4f} | Macro-F1 {mf1:.4f} | Weighted-F1 {wf1:.4f}')
    print(classification_report(true_names,pred_names,zero_division=0))

    # Per-domain F1
    tc = test_df.reset_index(drop=True).copy()
    tc['pred'] = pred_names; tc['true'] = true_names
    tc['domain'] = tc['source_file'].str.replace('.csv','',regex=False).str.replace('_',' ').str.title()
    domain_rows = []
    print(f'\n  {"Domain":<45} {"N":>4}  {"Macro-F1":>9}  {"Acc":>6}')
    for domain in sorted(tc['domain'].unique()):
        sub   = tc[tc['domain']==domain]
        dm_f1 = f1_score(sub['true'],sub['pred'],average='macro',zero_division=0)
        dm_acc= accuracy_score(sub['true'],sub['pred'])
        domain_rows.append({'model':model_key,'task':task_name,'domain':domain,
                            'n':len(sub),'macro_f1':round(dm_f1,4),'accuracy':round(dm_acc,4)})
        print(f'  {domain:<45} {len(sub):>4}  {dm_f1:>9.4f}  {dm_acc:>6.4f}')

    # Confusion matrix
    cm  = confusion_matrix(true_names,pred_names,labels=label_encoder.classes_)
    n   = num_labels
    fig,ax = plt.subplots(figsize=(max(8,n*0.55),max(6,n*0.45)))
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',linewidths=0.4,
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_,ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'{task_name} — {model_key}',fontweight='bold')
    plt.xticks(rotation=45,ha='right',fontsize=7)
    plt.yticks(fontsize=7)
    plt.tight_layout(); plt.show(); plt.close(fig)

    del model,trainer,train_ds,dev_ds,test_ds
    torch.cuda.empty_cache(); gc.collect()
    return {
        'overall':{'model':model_key,'task':task_name,'accuracy':round(acc,4),
                   'macro_f1':round(mf1,4),'weighted_f1':round(wf1,4)},
        'per_domain':domain_rows
    }


def train_slot_model(model_name, model_key, bio_train, bio_dev, bio_test,
                     tag2id, id2tag, device='cuda', seed=42):
    """Fine-tune a token classification model for slot filling."""
    print(f'\n{"="*60}\n  Slot Filling | {model_key}\n{"="*60}')
    num_tags  = len(tag2id)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModelForTokenClassification.from_pretrained(
        model_name,num_labels=num_tags,id2label=id2tag,label2id=tag2id,
        ignore_mismatched_sizes=True).to(device)
    train_ds = SlotDataset(bio_train,tokenizer,tag2id)
    dev_ds   = SlotDataset(bio_dev,  tokenizer,tag2id)
    test_ds  = SlotDataset(bio_test, tokenizer,tag2id)

    def compute_metrics(eval_pred):
        logits,labels = eval_pred
        preds = np.argmax(logits,axis=-1)
        ts,ps = [],[]
        for pr,lr in zip(preds,labels):
            t,p = [],[]
            for pi,li in zip(pr,lr):
                if li==-100: continue
                t.append(id2tag[li]); p.append(id2tag[pi])
            ts.append(t); ps.append(p)
        return {'slot_f1':seq_f1(ts,ps,zero_division=0)}

    args = TrainingArguments(
        output_dir=f'./results/slot_{model_key}',
        num_train_epochs=10,per_device_train_batch_size=32,
        per_device_eval_batch_size=64,warmup_ratio=0.1,weight_decay=0.01,
        learning_rate=2e-5,eval_strategy='epoch',save_strategy='no',
        load_best_model_at_end=False,logging_steps=20,seed=seed,
        report_to='none',fp16=(device=='cuda'),
    )
    trainer = Trainer(model=model,args=args,train_dataset=train_ds,
                      eval_dataset=dev_ds,compute_metrics=compute_metrics,
                      data_collator=DataCollatorForTokenClassification(tokenizer))
    trainer.train()
    preds_out = trainer.predict(test_ds)
    preds = np.argmax(preds_out.predictions,axis=-1)
    ts,ps = [],[]
    for pr,lr in zip(preds,preds_out.label_ids):
        t,p = [],[]
        for pi,li in zip(pr,lr):
            if li==-100: continue
            t.append(id2tag[li]); p.append(id2tag[pi])
        ts.append(t); ps.append(p)
    sf1 = seq_f1(ts,ps,zero_division=0)
    print(f'\n  Slot F1 (seqeval): {sf1:.4f}')
    print(seq_report(ts,ps,zero_division=0))
    del model,trainer,train_ds,dev_ds,test_ds
    torch.cuda.empty_cache(); gc.collect()
    return {'model':model_key,'task':'Slot Filling','slot_f1':round(sf1,4)}


In [ ]:
%%writefile /kaggle/working/swahili_llm.py
"""Zero-shot LLM evaluation utilities."""
import json, re
from sklearn.metrics import accuracy_score, f1_score


def make_system_prompt(intent_list, tone_list):
    return (
        'You are an NLU classifier for a Swahili virtual assistant.\n'
        'Given a Swahili utterance, output ONLY a valid JSON object with two keys:\n'
        f'  - "intent": one of {intent_list}\n'
        f'  - "tone": one of {tone_list}\n'
        'Do not include any explanation. Output only the JSON object.'
    )


def _parse_response(text):
    raw = re.sub(r'```json|```','',text).strip()
    data = json.loads(raw)
    return data.get('intent','unknown'), data.get('tone','unknown')


def predict_claude(client, utterance, system_prompt):
    try:
        msg = client.messages.create(
            model='claude-sonnet-4-5',max_tokens=100,
            system=system_prompt,
            messages=[{'role':'user','content':utterance}]
        )
        return _parse_response(msg.content[0].text)
    except Exception as e:
        print(f'  ⚠ Claude: {e}')
        return 'unknown','unknown'


def predict_gpt(client, utterance, system_prompt):
    try:
        resp = client.chat.completions.create(
            model='gpt-4o-mini',max_tokens=100,
            messages=[{'role':'system','content':system_prompt},
                      {'role':'user','content':utterance}]
        )
        return _parse_response(resp.choices[0].message.content)
    except Exception as e:
        print(f'  ⚠ GPT: {e}')
        return 'unknown','unknown'


def predict_gemini(model, utterance, system_prompt):
    try:
        resp = model.generate_content(f'{system_prompt}\n\nUtterance: {utterance}')
        return _parse_response(resp.text)
    except Exception as e:
        print(f'  ⚠ Gemini: {e}')
        return 'unknown','unknown'


def run_llm_evaluation(llm_registry, test_df, intent_list, tone_list, n_samples=100, seed=42):
    """Run zero-shot evaluation for all LLMs. Returns list of result dicts."""
    sample = test_df.sample(n=min(n_samples,len(test_df)),random_state=seed).reset_index(drop=True)
    true_intents = sample['intent'].tolist()
    true_tones   = sample['tone'].tolist()
    results = []
    for model_name, predict_fn in llm_registry.items():
        print(f'\n{"="*55}\n  {model_name}\n{"="*55}')
        pi_list, pt_list = [], []
        for i,row in sample.iterrows():
            pi,pt = predict_fn(row['utterance'])
            pi_list.append(pi); pt_list.append(pt)
            if (i+1)%20==0: print(f'  {i+1}/{len(sample)} done...')
        i_acc = accuracy_score(true_intents,pi_list)
        i_f1  = f1_score(true_intents,pi_list,average='macro',
                         zero_division=0,labels=intent_list)
        t_acc = accuracy_score(true_tones,pt_list)
        t_f1  = f1_score(true_tones,pt_list,average='macro',
                         zero_division=0,labels=tone_list)
        print(f'  Intent  Acc {i_acc:.4f} | Macro-F1 {i_f1:.4f}')
        print(f'  Tone    Acc {t_acc:.4f} | Macro-F1 {t_f1:.4f}')
        results.extend([
            {'model':model_name,'task':'Intent','accuracy':round(i_acc,4),
             'macro_f1':round(i_f1,4),'weighted_f1':'—'},
            {'model':model_name,'task':'Tone','accuracy':round(t_acc,4),
             'macro_f1':round(t_f1,4),'weighted_f1':'—'},
        ])
    return results


In [ ]:
%%writefile /kaggle/working/swahili_eval.py
"""Evaluation, visualisation and LaTeX generation utilities."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def per_domain_pivot(domain_results, test_df, task='Intent'):
    """Build model x domain Macro-F1 pivot table."""
    df   = pd.DataFrame([r for r in domain_results if r['task']==task])
    piv  = df.pivot_table(index='domain',columns='model',values='macro_f1').round(4)
    sizes = (test_df['source_file']
             .str.replace('.csv','',regex=False)
             .str.replace('_',' ').str.title()
             .value_counts().rename('test_n'))
    return piv.join(sizes).sort_values('test_n',ascending=False)


def plot_per_domain(pivot, title='Per-Domain Macro-F1'):
    models = [m for m in ['mBERT','XLM-R','AfroXLMR'] if m in pivot.columns]
    colors = ['#534AB7','#1D9E75','#E8593C']
    x = np.arange(len(pivot)); w = 0.25
    fig,ax = plt.subplots(figsize=(14,6))
    for i,(m,c) in enumerate(zip(models,colors)):
        ax.bar(x+i*w,pivot[m],w,label=m,color=c,alpha=0.85,edgecolor='white')
    ax.set_xticks(x+w); ax.set_xticklabels(pivot.index,rotation=35,ha='right',fontsize=9)
    ax.set_ylabel('Macro-F1'); ax.set_ylim(0,1.0)
    ax.set_title(title,fontweight='bold'); ax.legend()
    overall = pd.DataFrame([r for r in []]).get('macro_f1',pd.Series())
    plt.tight_layout(); plt.show(); plt.close(fig)


def plot_summary(all_results):
    tasks   = ['Intent','Tone','Slot Filling']
    metrics = ['macro_f1','macro_f1','slot_f1']
    colors  = ['#534AB7','#1D9E75','#E8593C']
    fig,axes = plt.subplots(1,3,figsize=(16,5))
    for ax,task,met,col in zip(axes,tasks,metrics,colors):
        sub = all_results[all_results['task']==task].copy()
        if met not in sub.columns: continue
        sub[met] = pd.to_numeric(sub[met],errors='coerce')
        sub = sub.dropna(subset=[met])
        bars = ax.bar(sub['model'],sub[met],color=col,alpha=0.85,edgecolor='white')
        for bar,val in zip(bars,sub[met]):
            ax.text(bar.get_x()+bar.get_width()/2,val+0.005,
                    f'{val:.3f}',ha='center',va='bottom',fontsize=8)
        ax.set_title(task,fontweight='bold'); ax.set_ylim(0,1.05)
        ax.set_ylabel('Score'); ax.tick_params(axis='x',rotation=25)
    plt.suptitle('SwahiliNLU Baseline Results',fontsize=13,fontweight='bold',y=1.02)
    plt.tight_layout(); plt.show(); plt.close(fig)


def build_latex_tables(all_results, slot_results):
    """Generate LaTeX Table 3 (classification) and Table 4 (slots)."""
    clf = all_results[all_results['task'].isin(['Intent','Tone'])]
    ft  = clf[clf['model'].isin(['mBERT','XLM-R','AfroXLMR'])]
    zs  = clf[~clf['model'].isin(['mBERT','XLM-R','AfroXLMR'])]
    best_i = ft[ft['task']=='Intent']['macro_f1'].max()
    best_t = ft[ft['task']=='Tone']['macro_f1'].max()
    best_zi= zs[zs['task']=='Intent']['macro_f1'].max()
    best_zt= zs[zs['task']=='Tone']['macro_f1'].max()

    def bold(val,best):
        return f'\\textbf{{{val}}}' if str(val)==str(best) else str(val)

    t3 = ['\\begin{table}[t]',
          '\\caption{Classification results. Bold=best per task/category.}',
          '\\label{tab:clf}\\setlength{\\tabcolsep}{3pt}\\small',
          '\\begin{tabular}{llccc}\\toprule',
          '\\textbf{Model} & \\textbf{Task} & \\textbf{Acc.} & \\textbf{Mac-F1} & \\textbf{Wt-F1} \\\\',
          '\\midrule\\multicolumn{5}{l}{\\textit{Fine-tuned models}} \\\\']
    for _,r in ft.iterrows():
        b = best_i if r['task']=='Intent' else best_t
        t3.append(f"  {r['model']} & {r['task']} & {r['accuracy']} & "
                  f"{bold(r['macro_f1'],b)} & {r['weighted_f1']} \\\\")
    t3 += ['\\midrule\\multicolumn{5}{l}{\\textit{Zero-shot LLMs}} \\\\']
    for _,r in zs.iterrows():
        b = best_zi if r['task']=='Intent' else best_zt
        t3.append(f"  {r['model']} & {r['task']} & {r['accuracy']} & "
                  f"{bold(r['macro_f1'],b)} & --- \\\\")
    t3 += ['\\bottomrule\\end{tabular}\\end{table}']

    sdf   = pd.DataFrame(slot_results)
    best_s= sdf['slot_f1'].max()
    t4 = ['\\begin{table}[t]',
          '\\caption{Slot filling results (seqeval Macro-F1).}',
          '\\label{tab:slots}\\small',
          '\\begin{tabular}{lc}\\toprule',
          '\\textbf{Model} & \\textbf{Slot F1} \\\\',' \\midrule']
    for _,r in sdf.iterrows():
        b = '\\textbf{' if r['slot_f1']==best_s else ''
        e = '}' if r['slot_f1']==best_s else ''
        t4.append(f'  {b}{r["model"]}{e} & {b}{r["slot_f1"]}{e} \\\\')
    t4 += ['\\bottomrule\\end{tabular}\\end{table}']
    return '\n'.join(t3), '\n'.join(t4)


In [ ]:
# ── Verify all modules written ───────────────────────────────────────────────
import os, sys
sys.path.insert(0, '/kaggle/working')
modules = ['swahili_data','swahili_datasets','swahili_train','swahili_llm','swahili_eval']
for m in modules:
    path = f'/kaggle/working/{m}.py'
    size = os.path.getsize(path)
    print(f'  ✓ {m}.py  ({size:,} bytes)')
print('\nAll modules ready.')

---
## 3. Load Data <a id='3'></a>

In [ ]:
from swahili_data import load_dataset

# Raw line counts
import glob
csv_files = sorted(glob.glob(f'{DATA_DIR}/*.csv'))
print(f'  {"File":<40} {"Raw rows":>8}')
total = 0
for f in csv_files:
    from pathlib import Path
    fname = Path(f).name
    with open(f,'r',encoding='utf-8') as fh:
        n = sum(1 for _ in fh) - 1
    total += n
    print(f'  {fname:<40} {n:>8}')
print(f'  {"TOTAL":<40} {total:>8}')

# Load and clean
df = load_dataset(DATA_DIR)
print(f'\nDomains loaded  : {df["source_file"].nunique()}')
print(f'Tone distribution:')
print(df['tone'].value_counts().to_string())

---
## 4. Train / Dev / Test Split <a id='4'></a>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Drop intents with < 3 samples
ic    = df['intent'].value_counts()
rare  = ic[ic<3].index.tolist()
if rare:
    print(f'Dropping {len(rare)} rare intents: {rare}')
    df_split = df[~df['intent'].isin(rare)].reset_index(drop=True)
else:
    df_split = df.copy()

train_df, temp_df = train_test_split(df_split,test_size=0.30,
                                     random_state=SEED,stratify=df_split['intent'])

# Drop intents with <2 samples in temp before second split
temp_ic = temp_df['intent'].value_counts()
valid   = temp_ic[temp_ic>=2].index
temp_df = temp_df[temp_df['intent'].isin(valid)]

dev_df, test_df = train_test_split(temp_df,test_size=0.667,
                                   random_state=SEED,stratify=temp_df['intent'])

intent_enc  = LabelEncoder().fit(df_split['intent'])
tone_enc    = LabelEncoder().fit(df_split['tone'])
NUM_INTENTS = len(intent_enc.classes_)
NUM_TONES   = len(tone_enc.classes_)

print(f'Train:{len(train_df):,}  Dev:{len(dev_df):,}  Test:{len(test_df):,}')
print(f'Intents:{NUM_INTENTS}  Tones:{NUM_TONES}')
print(f'\nDomains in test set:')
print(test_df['source_file'].value_counts().sort_index().to_string())

---
## 5. Experiment 1 — Intent Classification <a id='5'></a>

In [ ]:
import shutil
from swahili_train import train_classifier

if os.path.exists('./results'): shutil.rmtree('./results')

intent_results, intent_domain_results = [], []
for model_key, model_name in MODELS.items():
    result = train_classifier(model_name, model_key,
                              train_df, dev_df, test_df,
                              intent_enc, NUM_INTENTS,
                              task_name='Intent', device=DEVICE, seed=SEED)
    intent_results.append(result['overall'])
    intent_domain_results.extend(result['per_domain'])

import pandas as pd
pd.DataFrame(intent_results)

---
## 6. Experiment 2 — Tone Classification <a id='6'></a>

In [ ]:
tone_results, tone_domain_results = [], []
for model_key, model_name in MODELS.items():
    result = train_classifier(model_name, model_key,
                              train_df, dev_df, test_df,
                              tone_enc, NUM_TONES,
                              task_name='Tone', device=DEVICE, seed=SEED)
    tone_results.append(result['overall'])
    tone_domain_results.extend(result['per_domain'])

pd.DataFrame(tone_results)

---
## 7. Experiment 3 — Slot Filling <a id='7'></a>

In [ ]:
from swahili_datasets import build_bio_df
from swahili_train import train_slot_model

bio_train = build_bio_df(train_df)
bio_dev   = build_bio_df(dev_df)
bio_test  = build_bio_df(test_df)

all_tags = ['O'] + sorted({t for tags in bio_train['tags'] for t in tags if t!='O'})
tag2id   = {t:i for i,t in enumerate(all_tags)}
id2tag   = {i:t for t,i in tag2id.items()}

print(f'BIO label set: {len(tag2id)} tags')

slot_results = []
for model_key, model_name in MODELS.items():
    result = train_slot_model(model_name, model_key,
                              bio_train, bio_dev, bio_test,
                              tag2id, id2tag,
                              device=DEVICE, seed=SEED)
    slot_results.append(result)

pd.DataFrame(slot_results)

---
## 8. Experiment 4 — Zero-shot LLM Baseline <a id='8'></a>
> Add API keys: **Kaggle → Add-ons → Secrets** → `ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, `GEMINI_API_KEY`

In [ ]:
import anthropic, openai, google.generativeai as genai
from kaggle_secrets import UserSecretsClient
from swahili_llm import make_system_prompt, predict_claude, predict_gpt, predict_gemini, run_llm_evaluation

secrets = UserSecretsClient()
anthropic_client = anthropic.Anthropic(api_key=secrets.get_secret('ANTHROPIC_API_KEY'))
openai_client    = openai.OpenAI(api_key=secrets.get_secret('OPENAI_API_KEY'))
genai.configure(api_key=secrets.get_secret('GEMINI_API_KEY'))
gemini_model     = genai.GenerativeModel('gemini-2.5-flash')

INTENT_LIST   = sorted(df_split['intent'].unique().tolist())
TONE_LIST     = ['imperative','polite','conversational']
SYSTEM_PROMPT = make_system_prompt(INTENT_LIST, TONE_LIST)

# Bind API clients into predict functions
LLM_REGISTRY = {
    'Claude Sonnet (zero-shot)'    : lambda u: predict_claude(anthropic_client, u, SYSTEM_PROMPT),
    'GPT-4o mini (zero-shot)'      : lambda u: predict_gpt(openai_client, u, SYSTEM_PROMPT),
    'Gemini 2.5 Flash (zero-shot)' : lambda u: predict_gemini(gemini_model, u, SYSTEM_PROMPT),
}

llm_results = run_llm_evaluation(
    LLM_REGISTRY, test_df, INTENT_LIST, TONE_LIST, n_samples=100, seed=SEED
)
pd.DataFrame(llm_results)

---
## 9. Per-Domain Macro-F1 Analysis <a id='9'></a>

In [ ]:
from swahili_eval import per_domain_pivot, plot_per_domain

# Intent
pivot_intent = per_domain_pivot(intent_domain_results, test_df, task='Intent')
print('=== PER-DOMAIN INTENT MACRO-F1 ===')
print(pivot_intent.to_string())
pivot_intent.to_csv('per_domain_intent_f1.csv')
plot_per_domain(pivot_intent, title='Per-Domain Intent Macro-F1')

# Tone
pivot_tone = per_domain_pivot(tone_domain_results, test_df, task='Tone')
print('\n=== PER-DOMAIN TONE MACRO-F1 ===')
print(pivot_tone.to_string())
pivot_tone.to_csv('per_domain_tone_f1.csv')
plot_per_domain(pivot_tone, title='Per-Domain Tone Macro-F1')

---
## 10. Results Summary & LaTeX Tables <a id='10'></a>

In [ ]:
from swahili_eval import plot_summary, build_latex_tables

# Combine all results
all_results = pd.concat([
    pd.DataFrame(intent_results),
    pd.DataFrame(tone_results),
    pd.DataFrame(slot_results),
    pd.DataFrame(llm_results),
], ignore_index=True).fillna('—')

print('=== FINAL RESULTS ===')
print(all_results.to_string(index=False))
all_results.to_csv('results_final.csv', index=False)

# Visual summary
plot_summary(all_results)

# LaTeX tables
t3, t4 = build_latex_tables(all_results, slot_results)
print('\n--- TABLE 3 ---')
print(t3)
print('\n--- TABLE 4 ---')
print(t4)